In [ ]:
import sys
import os

sys.path.append("..")
sys.path.append("../..")
sys.path.append("../../..")

In [ ]:
from datasets import load_dataset, load_from_disk
if not os.path.exists("../../data/criteo_attribution_dataset"):
    ds = load_dataset("criteo/criteo-attribution-dataset")

    ds.save_to_disk("../../data/criteo_attribution_dataset")

In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
import gc

In [ ]:
raw_df = load_from_disk("../../data/criteo_attribution_dataset")["train"].with_format("pandas")[:]
raw_df = raw_df.sort_values("timestamp").reset_index(drop=True)
raw_df.head(), raw_df.shape

(   timestamp       uid  campaign  conversion  conversion_timestamp  \
 0          0  20073966  22589171           0                    -1   
 1          2  24607497    884761           0                    -1   
 2          2  28474333  18975823           0                    -1   
 3          3   7306395  29427842           1               1449193   
 4          3  25357769  13365547           0                    -1   
 
    conversion_id  attribution  click  click_pos  click_nb  ...  \
 0             -1            0      0         -1        -1  ...   
 1             -1            0      0         -1        -1  ...   
 2             -1            0      0         -1        -1  ...   
 3        3063962            0      1          0         7  ...   
 4             -1            0      0         -1        -1  ...   
 
    time_since_last_click      cat1      cat2      cat3      cat4      cat5  \
 0                     -1   5824233   9312274   3490278  29196072  11409686   
 1        

In [4]:
raw_df = raw_df.dropna().reset_index(drop=True)
raw_df.head(), raw_df.shape

(   timestamp       uid  campaign  conversion  conversion_timestamp  \
 0          0  20073966  22589171           0                    -1   
 1          2  24607497    884761           0                    -1   
 2          2  28474333  18975823           0                    -1   
 3          3   7306395  29427842           1               1449193   
 4          3  25357769  13365547           0                    -1   
 
    conversion_id  attribution  click  click_pos  click_nb  ...  \
 0             -1            0      0         -1        -1  ...   
 1             -1            0      0         -1        -1  ...   
 2             -1            0      0         -1        -1  ...   
 3        3063962            0      1          0         7  ...   
 4             -1            0      0         -1        -1  ...   
 
    time_since_last_click      cat1      cat2      cat3      cat4      cat5  \
 0                     -1   5824233   9312274   3490278  29196072  11409686   
 1        

In [5]:
raw_df['is_first_click'] = (raw_df['time_since_last_click'] == -1).astype(int)

In [6]:
train_len = 0.7 * len(raw_df)
val_len = 0.1 * len(raw_df)

idx_train_end = int(train_len)
idx_val_end = int(train_len + val_len)

df_train = raw_df.iloc[:idx_train_end].copy()
df_val = raw_df.iloc[idx_train_end:idx_val_end].copy()
df_bidding = raw_df.iloc[idx_val_end:].copy()

In [7]:
def train_model_pair(df_train, df_val, config):
	print(f"Desc: {config['desc']}\n{'='*30}")

	selected_features = config['features']
	current_cat_features = config['cat_features']
	params = config['params']

	if config['sample_rate'] < 1.0:
		n_rows = int(len(df_train) * config['sample_rate'])
		df_t = df_train.iloc[-n_rows:]
	else:
		df_t = df_train

	print(f"Features: {len(selected_features)} | Cat features: {len(current_cat_features)}")
	print(f"Train rows: {len(df_t)}")

	metrics = {}

	print("--> Fitting CVR Model (Clicks only)...")

	df_t_clicks = df_t[df_t['click'] == 1]
	df_v_clicks = df_val[df_val['click'] == 1]

	train_pool = Pool(df_t_clicks[selected_features], df_t_clicks['conversion'], cat_features=current_cat_features)
	val_pool = Pool(df_v_clicks[selected_features], df_v_clicks['conversion'], cat_features=current_cat_features)

	model_cvr = CatBoostClassifier(**params)
	model_cvr.fit(train_pool, eval_set=val_pool)

	preds = model_cvr.predict_proba(df_v_clicks[selected_features])[:, 1]
	metrics['cvr_best_score'] = model_cvr.get_best_score()['validation']['Logloss']
	metrics['cvr_auc'] = roc_auc_score(df_v_clicks['conversion'], preds)
	metrics['cvr_pr'] = average_precision_score(df_v_clicks['conversion'], preds)

	del df_t_clicks, df_v_clicks, train_pool, val_pool, preds
	gc.collect()

	print("--> Fitting CTR Model...")

	train_pool = Pool(df_t[selected_features], df_t['click'], cat_features=current_cat_features)
	val_pool = Pool(df_val[selected_features], df_val['click'], cat_features=current_cat_features)

	model_ctr = CatBoostClassifier(**params)
	model_ctr.fit(train_pool, eval_set=val_pool)

	val_sub = df_val.sample(n=min(1_000_000, len(df_val)), random_state=42)
	preds = model_ctr.predict_proba(val_sub[selected_features])[:, 1]
	metrics['ctr_best_score'] = model_ctr.get_best_score()['validation']['Logloss']
	metrics['ctr_roc_auc'] = roc_auc_score(val_sub['click'], preds)
	metrics['ctr_pr_auc'] = average_precision_score(val_sub['click'], preds)

	del train_pool, val_pool, preds, val_sub, df_t
	gc.collect()
	
	return {
		'ctr_model': model_ctr,
		'cvr_model': model_cvr,
		'metrics': metrics
	}

In [10]:
def apply_models_and_save(models_dict, df_bidding, name, config, filename="bidding_predictions.csv", ens_count=20):
    features = config['features']
    cat_features = config['cat_features']
    target_map = {'ctr': 'click', 'cvr': 'conversion'}

    pool = Pool(df_bidding[features], cat_features=cat_features)

    for m_type, target_col in target_map.items():
        print(f"--> Predicting {m_type}...")
        model = models_dict[f'{m_type}_model']

        virt_preds = model.virtual_ensembles_predict(
            pool, prediction_type='VirtEnsembles', virtual_ensembles_count=ens_count
        )

        mean_logit = virt_preds.mean(axis=1)
        std_logit = virt_preds.std(axis=1)

        mean_prob = 1 / (1 + np.exp(-mean_logit))

        sigma_data = mean_prob * (1 - mean_prob)

        df_bidding[f'{name}_{m_type}_pred_logit'] = mean_logit.astype(np.float32)
        df_bidding[f'{name}_{m_type}_logit_sigma'] = std_logit.astype(np.float32)
        df_bidding[f'{name}_{m_type}_sigma_data'] = sigma_data.astype(np.float32)

        del mean_logit, std_logit, mean_prob, sigma_data
        gc.collect()

    df_bidding.to_csv(filename, index=False)
    print(f"Saved to {filename}")
    return df_bidding

In [11]:
configs = {
	"all_features": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 1.0,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 300,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e2,
			'random_strength': 5.0,

			'posterior_sampling': True,
        }
    },
    "all_features_0_3": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 0.3,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 300,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e2,
			'random_strength': 3.0,

			'posterior_sampling': True,
        }
	},
    "all_features_0_1": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 0.1,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 4000,
			'early_stopping_rounds': 250,
			'learning_rate': 2e-2,

			'depth': 6,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 5e1,
			'random_strength': 2.0,

			'posterior_sampling': True,
        }
	},
    "all_features_0_03": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 0.03,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
    "all_features_0_01": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 0.01,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
    "all_features_0_003": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 0.003,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
    "all_features_0_001": {
		"desc": "All features, full data",
		"features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "time_since_last_click", "cat2", "cat4", "cat5", "is_first_click"],
		"cat_features": ["cat3", "cat8", "cat9", "cat6", "cat1", "campaign", "cat7", "cat2", "cat4", "cat5", "is_first_click"],
		"sample_rate": 0.001,
        "params": {
			'task_type': "CPU",
			'verbose': 200,
			'random_seed': 42,
			'allow_writing_files': False,

			'loss_function': 'Logloss',
			'eval_metric': 'Logloss',
			'boost_from_average': True,

			'iterations': 3000,
			'early_stopping_rounds': 200,
			'learning_rate': 2e-2,

			'depth': 4,
			'max_ctr_complexity': 4,
			'l2_leaf_reg': 1e1,
			'random_strength': 1.0,

			'posterior_sampling': True,
        }
	},
}

In [12]:
# del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features_0_001"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features_0_001", configs["all_features_0_001"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 11527
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.4146762	test: 0.3923494	best: 0.3923494 (0)	total: 93.2ms	remaining: 4m 39s
200:	learn: 0.3237024	test: 0.3025918	best: 0.3025918 (200)	total: 3.17s	remaining: 44.2s
400:	learn: 0.3137561	test: 0.3015558	best: 0.3015408 (397)	total: 7.32s	remaining: 47.5s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3015408489
bestIteration = 397

Shrink model to first 398 iterations.
--> Fitting CTR Model...
0:	learn: 0.6562193	test: 0.6570187	best: 0.6570187 (0)	total: 115ms	remaining: 5m 43s
200:	learn: 0.6149041	test: 0.6216256	best: 0.6216256 (200)	total: 20.6s	remaining: 4m 46s
400:	learn: 0.6096965	test: 0.6195979	best: 0.6195977 (399)	total: 43.5s	remaining: 4m 42s
600:	learn: 0.6058520	test: 0.6188744	best: 0.6188733 (599)	total: 1m 7s	remaining: 4m 28s
800:	learn: 0.6025074	test: 0.6185551	best: 0.6185551 (799)	total: 1m 31s	remaining: 4m 1

{'cvr_best_score': 0.3015408489214851,
 'cvr_auc': 0.8194980556418188,
 'cvr_pr': 0.49862901497252105,
 'ctr_best_score': 0.6183631724282085,
 'ctr_roc_auc': 0.6600764358023271,
 'ctr_pr_auc': 0.5365660647072604}

In [13]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features_0_003"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features_0_003", configs["all_features_0_003"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 34582
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.4252047	test: 0.3928716	best: 0.3928716 (0)	total: 160ms	remaining: 7m 58s
200:	learn: 0.3302364	test: 0.2994727	best: 0.2994727 (200)	total: 8.85s	remaining: 2m 3s
400:	learn: 0.3251295	test: 0.2979892	best: 0.2979830 (397)	total: 19.9s	remaining: 2m 9s
600:	learn: 0.3218279	test: 0.2978165	best: 0.2977589 (500)	total: 33.8s	remaining: 2m 14s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.2977588617
bestIteration = 500

Shrink model to first 501 iterations.
--> Fitting CTR Model...
0:	learn: 0.6538419	test: 0.6570659	best: 0.6570659 (0)	total: 338ms	remaining: 16m 52s
200:	learn: 0.6079651	test: 0.6166746	best: 0.6166746 (200)	total: 22.5s	remaining: 5m 13s
400:	learn: 0.6045555	test: 0.6148413	best: 0.6148394 (399)	total: 48.7s	remaining: 5m 15s
600:	learn: 0.6027077	test: 0.6141570	best: 0.6141570 (600)	total: 1m 18s	remaining: 5m 1

{'cvr_best_score': 0.297758861737492,
 'cvr_auc': 0.8288880954575815,
 'cvr_pr': 0.5117238574722525,
 'ctr_best_score': 0.613182858855234,
 'ctr_roc_auc': 0.6702149645609619,
 'ctr_pr_auc': 0.5471039187350838}

In [14]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features_0_01"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features_0_01", configs["all_features_0_01"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 115276
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.4111821	test: 0.3918917	best: 0.3918917 (0)	total: 181ms	remaining: 9m 2s
200:	learn: 0.3099509	test: 0.2946603	best: 0.2946603 (200)	total: 10.3s	remaining: 2m 23s
400:	learn: 0.3064737	test: 0.2939345	best: 0.2939334 (396)	total: 24.4s	remaining: 2m 38s
600:	learn: 0.3045408	test: 0.2936862	best: 0.2936811 (589)	total: 37.8s	remaining: 2m 30s
800:	learn: 0.3032739	test: 0.2937918	best: 0.2936576 (622)	total: 53.8s	remaining: 2m 27s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.2936575984
bestIteration = 622

Shrink model to first 623 iterations.
--> Fitting CTR Model...
0:	learn: 0.6601572	test: 0.6569033	best: 0.6569033 (0)	total: 299ms	remaining: 14m 57s
200:	learn: 0.6124523	test: 0.6122595	best: 0.6122595 (200)	total: 24.7s	remaining: 5m 43s
400:	learn: 0.6097004	test: 0.6102470	best: 0.6102470 (400)	total: 58s	remaining: 6m 15

{'cvr_best_score': 0.29365759841852995,
 'cvr_auc': 0.8352936030990691,
 'cvr_pr': 0.5214771497679982,
 'ctr_best_score': 0.6093077392131964,
 'ctr_roc_auc': 0.6782980270495929,
 'ctr_pr_auc': 0.5548518458205726}

In [15]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features_0_03"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features_0_03", configs["all_features_0_03"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 345828
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.4259796	test: 0.4251047	best: 0.4251047 (0)	total: 326ms	remaining: 16m 16s
200:	learn: 0.2985075	test: 0.2991796	best: 0.2991796 (200)	total: 13s	remaining: 3m 1s
400:	learn: 0.2918524	test: 0.2933313	best: 0.2933313 (400)	total: 29.4s	remaining: 3m 10s
600:	learn: 0.2895151	test: 0.2919243	best: 0.2919243 (600)	total: 46.3s	remaining: 3m 5s
800:	learn: 0.2880381	test: 0.2915169	best: 0.2915147 (793)	total: 1m 2s	remaining: 2m 50s
1000:	learn: 0.2871904	test: 0.2913660	best: 0.2913633 (998)	total: 1m 20s	remaining: 2m 40s
1200:	learn: 0.2866912	test: 0.2913062	best: 0.2913020 (1188)	total: 1m 36s	remaining: 2m 25s
1400:	learn: 0.2862539	test: 0.2912706	best: 0.2912623 (1355)	total: 1m 53s	remaining: 2m 9s
1600:	learn: 0.2858743	test: 0.2912790	best: 0.2912554 (1486)	total: 2m 9s	remaining: 1m 53s
Stopped by overfitting detector  (200 iterations wait)

bes

{'cvr_best_score': 0.2912553758116787,
 'cvr_auc': 0.8404810143454426,
 'cvr_pr': 0.5248964847076809,
 'ctr_best_score': 0.6049890883492266,
 'ctr_roc_auc': 0.6863952163054338,
 'ctr_pr_auc': 0.5624591098615526}

In [16]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features_0_1"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features_0_1", configs["all_features_0_1"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 1152761
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3964083	test: 0.3906791	best: 0.3906791 (0)	total: 374ms	remaining: 24m 54s
200:	learn: 0.2897150	test: 0.2886504	best: 0.2886504 (200)	total: 40.5s	remaining: 12m 44s
400:	learn: 0.2860632	test: 0.2865484	best: 0.2865484 (400)	total: 1m 32s	remaining: 13m 50s
600:	learn: 0.2846290	test: 0.2859015	best: 0.2859015 (600)	total: 2m 27s	remaining: 13m 54s
800:	learn: 0.2833404	test: 0.2854651	best: 0.2854623 (796)	total: 3m 24s	remaining: 13m 37s
1000:	learn: 0.2826222	test: 0.2853442	best: 0.2853434 (998)	total: 4m 21s	remaining: 13m 4s
1200:	learn: 0.2821688	test: 0.2852774	best: 0.2852763 (1199)	total: 5m 20s	remaining: 12m 26s
1400:	learn: 0.2818036	test: 0.2852647	best: 0.2852589 (1307)	total: 6m 18s	remaining: 11m 42s
Stopped by overfitting detector  (250 iterations wait)

bestTest = 0.2852589077
bestIteration = 1307

Shrink model to first 1308 iteratio

{'cvr_best_score': 0.2852589077168007,
 'cvr_auc': 0.8477403371183911,
 'cvr_pr': 0.5468950594673894,
 'ctr_best_score': 0.601861384392886,
 'ctr_roc_auc': 0.691908538022605,
 'ctr_pr_auc': 0.5683571763767136}

In [ ]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features_0_3"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features_0_3", configs["all_features_0_3"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 3458285
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3967949	test: 0.3909524	best: 0.3909524 (0)	total: 634ms	remaining: 42m 15s
200:	learn: 0.2883342	test: 0.2874652	best: 0.2874652 (200)	total: 1m 32s	remaining: 29m 10s
400:	learn: 0.2843743	test: 0.2849325	best: 0.2849325 (400)	total: 3m 28s	remaining: 31m 8s
600:	learn: 0.2829909	test: 0.2841201	best: 0.2841201 (600)	total: 5m 27s	remaining: 30m 51s
800:	learn: 0.2818235	test: 0.2834580	best: 0.2834580 (800)	total: 7m 27s	remaining: 29m 48s
1000:	learn: 0.2809430	test: 0.2830891	best: 0.2830891 (1000)	total: 9m 33s	remaining: 28m 37s
1200:	learn: 0.2803714	test: 0.2828662	best: 0.2828654 (1199)	total: 11m 39s	remaining: 27m 9s
1400:	learn: 0.2799275	test: 0.2827686	best: 0.2827686 (1400)	total: 13m 46s	remaining: 25m 33s
1600:	learn: 0.2796127	test: 0.2826988	best: 0.2826988 (1600)	total: 15m 56s	remaining: 23m 52s
1800:	learn: 0.2793580	test: 0.2826655

{'cvr_best_score': 0.28265837914554814,
 'cvr_auc': 0.8511629477310281,
 'cvr_pr': 0.5535385037733433,
 'ctr_best_score': 0.6004725337111614,
 'ctr_roc_auc': 0.6947928512384038,
 'ctr_pr_auc': 0.5717626610107649}

: 

In [14]:
del models
gc.collect()
models = train_model_pair(df_train, df_val, configs["all_features"])
df_bidding = apply_models_and_save(models, df_bidding, "all_features", configs["all_features"], filename="bidding_predictions.csv")
models['metrics']

Desc: All features, full data
Features: 12 | Cat features: 11
Train rows: 11527618
--> Fitting CVR Model (Clicks only)...
0:	learn: 0.3956298	test: 0.3910844	best: 0.3910844 (0)	total: 914ms	remaining: 1h 57s
200:	learn: 0.2887798	test: 0.2878887	best: 0.2878887 (200)	total: 2m 15s	remaining: 42m 39s
400:	learn: 0.2855020	test: 0.2855360	best: 0.2855360 (400)	total: 5m 9s	remaining: 46m 16s
600:	learn: 0.2842303	test: 0.2846659	best: 0.2846659 (600)	total: 8m 3s	remaining: 45m 36s
800:	learn: 0.2832811	test: 0.2840508	best: 0.2840508 (800)	total: 10m 54s	remaining: 43m 35s
1000:	learn: 0.2822968	test: 0.2834590	best: 0.2834590 (1000)	total: 13m 51s	remaining: 41m 31s
1200:	learn: 0.2817596	test: 0.2831883	best: 0.2831883 (1200)	total: 16m 51s	remaining: 39m 18s
1400:	learn: 0.2813835	test: 0.2830399	best: 0.2830399 (1400)	total: 19m 44s	remaining: 36m 37s
1600:	learn: 0.2811234	test: 0.2829813	best: 0.2829811 (1598)	total: 22m 40s	remaining: 33m 58s
1800:	learn: 0.2809360	test: 0.28293

{'cvr_best_score': 0.2829218548137628,
 'cvr_auc': 0.8513500260228819,
 'cvr_pr': 0.551811859518541,
 'ctr_best_score': 0.5993581652572881,
 'ctr_roc_auc': 0.6959822609792806,
 'ctr_pr_auc': 0.5733772355369997}